# Furniture detector (rotated boxes) — FloorPlanCAD · YOLO11m-OBB · 2×T4

Detects **bed, sofa, chair, table, wardrobe, cabinets, TV unit, toilet, sink, bath, stove, fridge, washing machine** with their **rotation**, so drawn furniture can be copied exactly into the 3D / Revit model, and the **kind of each door symbol** (single, double, sliding, unequal double) so the engine picks the right Revit door family.

**Before running (one time):** Accelerator **GPU T4 x2** · Internet **On** · Add-ons → Secrets → `HF_TOKEN` (write) attached ·
then **Save Version → Save & Run All (Commit)** (runs in the background; you can close the browser).

* First run converts the dataset (~30–45 min) and caches it to your private HF dataset repo; later runs skip that.
* Checkpoints (`last.pt`, `best.pt`, `results.csv`) are pushed to your private HF model repo every epoch.
* If a run is cut off, run the notebook again — it resumes from `last.pt`. A finished run is not retrained.
* Data: FloorPlanCAD (Voxel51 mirror) + ArchCAD-400K door / furniture symbols (gated: accept its access agreement once at https://huggingface.co/datasets/jackluoluo/ArchCAD with the HF_TOKEN account). Both CC BY-NC 4.0 — non-commercial.
* A run with a new `DATA_VERSION` (new classes) starts from the previous detector's weights instead of resuming.


In [ ]:
# ===================== CONFIG =====================
import os, sys, time, json, shutil, subprocess, threading, re
NOTEBOOK_START = time.time()
MODEL_REPO_NAME = "floorplan-furniture-obb"          # private HF model repo
DATA_REPO_NAME  = "floorplancad-obb-cache"           # private HF dataset repo (converted data)
BASE_MODEL      = "yolo11m-obb.pt"                   # pretrained rotated-box detector
IMGSZ, BATCH    = 1024, 16                           # 16 = 8 per GPU; auto-retries with 8 if it does not fit
MAX_EPOCHS      = 300                                # upper bound; the time budget decides the real length
TIME_BUDGET_HOURS = 11.25
VAL_FRACTION, SEED = 0.1, 42
DATA_VERSION    = "v2-doortypes"                     # v2: + door kinds (single / double / sliding / unequal double)
USE_ARCHCAD     = True                               # + ArchCAD-400K door / furniture symbols
ARCHCAD_LIMIT   = 15000                              # slices to convert (None = all ~40K)
DS_DIR   = "/tmp/furn_ds"                            # converted dataset (not saved as notebook output)
RUNS_DIR = "/tmp/furn_runs"
RUN_NAME = "obb"
SRC_DIR  = "/kaggle/working/furn_src"
os.makedirs(SRC_DIR, exist_ok=True)
print("config ok")

In [ ]:
!pip install -q "ultralytics==8.3.253" "huggingface_hub>=0.25" 2>&1 | grep -v "already satisfied" | tail -2

In [ ]:
%%writefile /kaggle/working/furn_src/furn_common.py
"""Furniture detector: FloorPlanCAD -> YOLO-OBB conversion, and tiled inference on plans."""
import json
import math
import os

import cv2
import numpy as np

# FloorPlanCAD classes we train on (where doors / windows / stairs / walls are comes from the floor-plan model)
FURN_CLASSES = ["bed", "sofa", "chair", "table", "bedside_cupboard", "tv_cabinet", "half_height_cabinet",
                "high_cabinet", "wardrobe", "sink", "bath", "bath_tub", "toilet", "squat_toilet", "urinal",
                "gas_stove", "refrigerator", "washing_machine"]
# door kinds the same detector learns (FloorPlanCAD + ArchCAD symbols). The floor-plan model says where a door
# is and how wide; the detector box on that door says which kind (picks the Revit family / IFC operation type)
DOOR_TYPES = ["single_door", "double_door", "sliding_door", "parent_child_door"]
DET_CLASSES = FURN_CLASSES + DOOR_TYPES
ZOOM_FACTORS = (0.75, 1.0, 1.4)
# typical real sizes (long side, metres) used to measure the drawing scale of the dataset
TYPICAL_LONG_M = {"bed": 2.0, "toilet": 0.7, "bath_tub": 1.7, "sofa": 2.0, "refrigerator": 0.7}


def norm_label(s):
    return str(s).strip().lower().replace(" ", "_").replace("-", "_")


def to_rgb_white(img):
    """PNG with alpha -> RGB on white (FloorPlanCAD renders are 4-channel)."""
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        a = img[:, :, 3:4].astype(np.float32) / 255.0
        return (img[:, :, :3].astype(np.float32) * a + 255.0 * (1 - a)).astype(np.uint8)
    return img


def det_to_obb(bbox, mask, W, H):
    """FiftyOne detection (normalised [x,y,w,h] + cropped mask) -> 4 corner points in pixels.
    The rotated rectangle is fitted to the drawn strokes, so it follows the furniture's rotation."""
    bx, by, bw, bh = [float(v) for v in bbox]
    x0, y0, w, h = bx * W, by * H, bw * W, bh * H
    pts = None
    if mask is not None:
        m = np.asarray(mask)
        if m.ndim == 3:
            m = m[..., 0]
        ys, xs = np.nonzero(m)
        if len(xs) >= 5:
            sx, sy = w / m.shape[1], h / m.shape[0]
            pts = np.stack([x0 + (xs + 0.5) * sx, y0 + (ys + 0.5) * sy], 1).astype(np.float32)
    if pts is None:
        pts = np.array([[x0, y0], [x0 + w, y0], [x0 + w, y0 + h], [x0, y0 + h]], np.float32)
    rect = cv2.minAreaRect(pts)
    (cw, ch) = rect[1]
    if cw < 2 or ch < 2:                     # degenerate -> axis-aligned box
        rect = ((x0 + w / 2, y0 + h / 2), (max(w, 2), max(h, 2)), 0.0)
    return cv2.boxPoints(rect), rect


def obb_line(cls_id, corners, W, H):
    c = np.clip(corners, 0, [W - 1, H - 1]) / [W, H]
    return "{} ".format(cls_id) + " ".join("{:.6f}".format(v) for v in c.reshape(-1))


def write_sample(img_path, dets, out_img, out_lbl, stats):
    """dets: list of (label, bbox, mask). Writes RGB jpg + YOLO-OBB label file. Returns #objects."""
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return 0
    img = to_rgb_white(img)
    H, W = img.shape[:2]
    lines = []
    for label, bbox, mask in dets:
        lab = norm_label(label)
        if lab not in DET_CLASSES:
            continue
        corners, rect = det_to_obb(bbox, mask, W, H)
        lines.append(obb_line(DET_CLASSES.index(lab), corners, W, H))
        stats.setdefault(lab, []).append(max(rect[1]))
    cv2.imwrite(out_img, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    with open(out_lbl, "w") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))
    return len(lines)


def dataset_px_per_m(stats):
    """Drawing scale of the dataset from typical object sizes (median of several classes)."""
    est = [np.median(stats[k]) / v for k, v in TYPICAL_LONG_M.items() if len(stats.get(k, [])) >= 20]
    return float(np.median(est)) if est else None


# ----------------------------------------------------------------------------- inference on a plan
def _onnx_names(sess):
    import ast
    return ast.literal_eval(sess.get_modelmeta().custom_metadata_map.get("names", "{}"))


def _obb_onnx(sess, crop_rgb, conf):
    """Ultralytics YOLO-OBB exported to ONNX: output (1, 4 + classes + 1, N) = cx, cy, w, h, class scores, angle (rad).
    Returns [(corners 4x2 in crop px, class id, score)]; overlaps are merged later by rotated_nms."""
    x = np.ascontiguousarray((crop_rgb.astype(np.float32) / 255.0).transpose(2, 0, 1)[None])
    out = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]          # (4 + nc + 1, N)
    nc = out.shape[0] - 5
    scores = out[4:4 + nc]
    cls = scores.argmax(0)
    sc = scores[cls, np.arange(scores.shape[1])]
    keep = np.nonzero(sc >= conf)[0]
    if len(keep) > 1:                                  # same NMS Ultralytics applies (rotated, per class)
        rects = [((float(out[0, k]), float(out[1, k])), (float(out[2, k]), float(out[3, k])),
                  float(np.degrees(out[4 + nc, k]))) for k in keep]
        sel = []
        for c_ in np.unique(cls[keep]):
            idx = [j for j, k in enumerate(keep) if cls[k] == c_]
            r = cv2.dnn.NMSBoxesRotated([rects[j] for j in idx], [float(sc[keep[j]]) for j in idx], conf, 0.7)
            sel += [idx[int(j)] for j in np.array(r).reshape(-1)]
        keep = keep[sorted(sel)]
    res = []
    for k in keep:
        cx, cy, w, h = out[:4, k]
        a = float(out[4 + nc, k])
        c, s_ = np.cos(a), np.sin(a)
        dx, dy = np.array([c, s_]) * w / 2, np.array([-s_, c]) * h / 2
        ctr = np.array([cx, cy])
        res.append((np.array([ctr + dx + dy, ctr + dx - dy, ctr - dx - dy, ctr - dx + dy]), int(cls[k]), float(sc[k])))
    return res


def detect_furniture(model, img_rgb, plan_px_per_m, train_px_per_m, conf=0.35, tile=1024, overlap=0.25,
                     zooms=None, roi=None):
    """Rescale the plan so furniture has its training size, run tiled OBB detection, merge.
    zooms: factors of the training scale (default ZOOM_FACTORS); roi: (x0, y0, x1, y1) in image px to search
    (the building), None = whole sheet. Tiles with almost no ink are skipped.
    Returns list of dicts in ORIGINAL image pixels: cls, conf, corners(4x2)."""
    names = model.names if hasattr(model, "names") else _onnx_names(model)
    ox, oy = 0, 0
    if roi is not None:
        Hi, Wi = img_rgb.shape[:2]
        x0_, y0_, x1_, y1_ = [int(round(v)) for v in roi]
        x0_, y0_, x1_, y1_ = max(x0_, 0), max(y0_, 0), min(x1_, Wi), min(y1_, Hi)
        if x1_ - x0_ > 16 and y1_ - y0_ > 16:
            img_rgb, ox, oy = img_rgb[y0_:y1_, x0_:x1_], x0_, y0_
    base = train_px_per_m / plan_px_per_m if (train_px_per_m and plan_px_per_m) else 1.0
    scales = [base * f for f in (zooms or ZOOM_FACTORS)]   # several zooms, merged below
    found = []
    for s in scales:
        s = float(np.clip(s, 0.2, 6.0))
        im = cv2.resize(img_rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
        H, W = im.shape[:2]
        step = int(tile * (1 - overlap))
        ys = list(range(0, max(H - tile, 0) + 1, step)) or [0]
        xs = list(range(0, max(W - tile, 0) + 1, step)) or [0]
        if ys[-1] + tile < H:
            ys.append(max(H - tile, 0))
        if xs[-1] + tile < W:
            xs.append(max(W - tile, 0))
        for y0 in ys:
            for x0 in xs:
                crop = np.full((tile, tile, 3), 255, np.uint8)
                c = im[y0:y0 + tile, x0:x0 + tile]
                if (c.min(axis=2) < 160).mean() < 0.002:       # blank paper: nothing to detect, skip the pass
                    continue
                crop[:c.shape[0], :c.shape[1]] = c
                if hasattr(model, "run"):                     # ONNX Runtime session
                    boxes = _obb_onnx(model, crop, conf)
                else:
                    res = model.predict(crop[:, :, ::-1], imgsz=tile, conf=conf, verbose=False)[0]
                    if res.obb is None or len(res.obb) == 0:
                        continue
                    pts_ = res.obb.xyxyxyxy.cpu().numpy()
                    boxes = [(pts_[k].reshape(4, 2), int(res.obb.cls[k]), float(res.obb.conf[k])) for k in range(len(pts_))]
                for p, ci, cf in boxes:
                    cx, cy = p[:, 0].mean(), p[:, 1].mean()
                    # ignore detections centred in the overlap margin of an inner tile edge
                    m = tile * overlap / 2
                    if (x0 > 0 and cx < m) or (y0 > 0 and cy < m) or \
                       (x0 + tile < W and cx > tile - m) or (y0 + tile < H and cy > tile - m):
                        continue
                    found.append({"cls": names[int(ci)], "conf": float(cf), "corners": (p + [x0, y0]) / s + [ox, oy]})
    return rotated_nms(found)


def rotated_nms(dets, iou_thr=0.4):
    from shapely.geometry import Polygon
    dets = sorted(dets, key=lambda d: -d["conf"])
    keep = []
    for d in dets:
        p = Polygon(d["corners"]).buffer(0)
        if p.area <= 0:
            continue
        ok = True
        for k in keep:
            q = k["_poly"]
            inter = p.intersection(q).area
            if inter / (p.area + q.area - inter + 1e-9) > iou_thr or inter / min(p.area, q.area) > 0.8:
                ok = False
                break
        if ok:
            d["_poly"] = p
            keep.append(d)
    for k in keep:
        k.pop("_poly", None)
    return keep


In [ ]:
%%writefile /kaggle/working/furn_src/fp_common.py
"""Shared code: classes, cached dataset + augmentation, model, losses, metrics."""
import json
import math
import os
import random

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(0)

# ----------------------------------------------------------------------------- classes
# Room head (from the official CubiCasa5K "rooms_selected" mapping)
ROOM_CLASSES = ["Background", "Outdoor", "Wall", "Kitchen", "LivingRoom", "Bedroom",
                "Bath", "Entry/Hallway", "Railing", "Storage", "Garage", "OtherRoom"]
# Icon head (official "icons_selected" mapping) + Stairs (parsed separately, id 11)
ICON_CLASSES = ["NoIcon", "Window", "Door", "Closet", "ElectricalAppliance", "Toilet",
                "Sink", "SaunaBench", "Fireplace", "Bathtub", "Chimney", "Stairs"]
# Junction heatmaps (orientation-free so flips/rotations stay valid)
HEAT_CLASSES = ["WallJunction", "OpeningEndpoint", "IconCorner"]

N_ROOM, N_ICON, N_HEAT = len(ROOM_CLASSES), len(ICON_CLASSES), len(HEAT_CLASSES)
N_OUT = N_ROOM + N_ICON + N_HEAT
IGNORE = 255
NOT_WALL = 254       # room target: "known not wall, room type unknown" (sources that only label walls)
# Rotation augmentation: plans with angled wings mostly use these angles; any angle is also trained
ROT_ANGLES = (30.0, 45.0, 60.0)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255.0
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255.0


# ----------------------------------------------------------------------------- heatmaps
def render_heatmaps(points, h, w, sigma):
    """points: (N,3) float array [x, y, type]. Returns (N_HEAT, h, w) float32, peak exactly 1."""
    heat = np.zeros((N_HEAT, h, w), dtype=np.float32)
    if points is None or len(points) == 0:
        return heat
    r = int(math.ceil(3 * sigma))
    ax = np.arange(-r, r + 1, dtype=np.float32)
    g = np.exp(-(ax[None, :] ** 2 + ax[:, None] ** 2) / (2 * sigma * sigma)).astype(np.float32)
    for x, y, t in points:
        cx, cy, t = int(round(float(x))), int(round(float(y))), int(t)
        if not (0 <= cx < w and 0 <= cy < h) or not (0 <= t < N_HEAT):
            continue
        x0, x1 = max(0, cx - r), min(w, cx + r + 1)
        y0, y1 = max(0, cy - r), min(h, cy + r + 1)
        patch = g[y0 - (cy - r):y1 - (cy - r), x0 - (cx - r):x1 - (cx - r)]
        np.maximum(heat[t, y0:y1, x0:x1], patch, out=heat[t, y0:y1, x0:x1])
    return heat


# ----------------------------------------------------------------------------- dataset
class CachedFloorplans(torch.utils.data.Dataset):
    """Reads the preprocessed cache: <root>/data/<id>_img.png, _lbl.png (B=room, G=icon), _pts.npy"""

    def __init__(self, root, split, train, crop=768, sigma=3.0, scale_range=(0.6, 1.8), rot_prob=0.6,
                 fixed_angle=None, sources=None, limit=None):
        """rot_prob: share of training crops rotated by a non-90-degree angle (30/45/60 +-3 deg, or any angle).
        fixed_angle: evaluation only - every sample rotated by exactly this angle (rotated-plan validation)."""
        with open(os.path.join(root, "index.json")) as f:
            index = json.load(f)
        self.items = index[split]
        if sources is not None:                     # e.g. ["cubicasa"]; entries without "source" are CubiCasa5K
            self.items = [it for it in self.items if it.get("source", "cubicasa") in sources]
        if limit:
            self.items = self.items[:limit]
        self.root = os.path.join(root, "data")
        self.train, self.crop, self.sigma, self.scale_range = train, crop, sigma, scale_range
        self.rot_prob, self.fixed_angle = rot_prob, fixed_angle

    def __len__(self):
        return len(self.items)

    def source_of(self, i):
        return self.items[i].get("source", "cubicasa")

    def _load(self, i):
        sid = self.items[i]["id"]
        img = cv2.imread(os.path.join(self.root, sid + "_img.png"), cv2.IMREAD_COLOR)
        lbl = cv2.imread(os.path.join(self.root, sid + "_lbl.png"), cv2.IMREAD_UNCHANGED)
        if img is None or lbl is None:
            raise RuntimeError(f"Corrupt cache entry {sid}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pp = os.path.join(self.root, sid + "_pts.npy")
        # sources without junction labels have no _pts file: their heat target is "unknown", not "none"
        pts = np.load(pp).astype(np.float32).reshape(-1, 3) if os.path.exists(pp) else None
        return img, lbl[:, :, 0].copy(), lbl[:, :, 1].copy(), pts

    def __getitem__(self, i):
        img, room, icon, pts = self._load(i)
        has_heat = pts is not None
        if pts is None:
            pts = np.zeros((0, 3), np.float32)
        if self.train:
            img, room, icon, pts = self._augment(img, room, icon, pts)
        elif self.fixed_angle is not None:
            img, room, icon, pts = self._rotate_whole(img, room, icon, pts, self.fixed_angle)
            img, room, icon = self._pad32(img, room, icon)
        else:
            img, room, icon = self._pad32(img, room, icon)
        h, w = room.shape
        heat = render_heatmaps(pts, h, w, self.sigma) if has_heat else np.full((N_HEAT, h, w), -1.0, np.float32)
        x = (img.astype(np.float32) - MEAN) / STD
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(room)).long(),
                torch.from_numpy(np.ascontiguousarray(icon)).long(),
                torch.from_numpy(heat))

    @staticmethod
    def _pad32(img, room, icon):
        h, w = room.shape
        ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
        if ph or pw:
            img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=(255, 255, 255))
            room = cv2.copyMakeBorder(room, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE)
            icon = cv2.copyMakeBorder(icon, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=IGNORE)
        return img, room, icon

    @staticmethod
    def _rotate_whole(img, room, icon, pts, angle):
        """Rotate a whole plan by `angle` degrees on a canvas that fits it (evaluation on rotated plans)."""
        h, w = room.shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        c, s_ = abs(M[0, 0]), abs(M[0, 1])
        W2, H2 = int(math.ceil(h * s_ + w * c)), int(math.ceil(h * c + w * s_))
        M[0, 2] += W2 / 2 - w / 2
        M[1, 2] += H2 / 2 - h / 2
        img = cv2.warpAffine(img, M, (W2, H2), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
        room = cv2.warpAffine(room, M, (W2, H2), flags=cv2.INTER_NEAREST, borderValue=IGNORE)
        icon = cv2.warpAffine(icon, M, (W2, H2), flags=cv2.INTER_NEAREST, borderValue=IGNORE)
        if len(pts):
            pts = pts.copy()
            xy = pts[:, :2] @ M[:, :2].T + M[:, 2]
            pts[:, 0], pts[:, 1] = xy[:, 0], xy[:, 1]
        return img, room, icon, pts

    def _augment(self, img, room, icon, pts):
        S = self.crop
        h, w = room.shape
        lo, hi = self.scale_range
        s = math.exp(random.uniform(math.log(lo), math.log(hi)))
        src = S / s  # crop size in source pixels
        x0 = random.uniform(0, w - src) if w > src else -random.uniform(0, src - w)
        y0 = random.uniform(0, h - src) if h > src else -random.uniform(0, src - h)
        M = np.array([[s, 0, -x0 * s], [0, s, -y0 * s]], dtype=np.float32)
        if random.random() < self.rot_prob:
            # angled walls: rotate the crop about its centre (the 90-degree turns / flips below add the rest,
            # so 30 also covers 120, 210, 300 and with the flip -30 ...). Corners outside the plan stay
            # white paper and are ignored in the loss.
            a = random.choice(ROT_ANGLES) + random.uniform(-3, 3) if random.random() < 0.7 else random.uniform(0, 90)
            R = cv2.getRotationMatrix2D((S / 2, S / 2), a, 1.0).astype(np.float32)
            M = (np.vstack([R, [0, 0, 1]]) @ np.vstack([M, [0, 0, 1]]))[:2].astype(np.float32)
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
        img = cv2.warpAffine(img, M, (S, S), flags=interp, borderMode=cv2.BORDER_CONSTANT,
                             borderValue=(255, 255, 255))
        room = cv2.warpAffine(room, M, (S, S), flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=IGNORE)
        icon = cv2.warpAffine(icon, M, (S, S), flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=IGNORE)
        if len(pts):
            pts = pts.copy()
            xy = pts[:, :2] @ M[:, :2].T + M[:, 2]
            pts[:, 0], pts[:, 1] = xy[:, 0], xy[:, 1]
        # rotations / flips (all heads are orientation-free, so labels stay valid)
        k = random.randint(0, 3)
        for _ in range(k):  # 90 deg counter-clockwise
            img, room, icon = np.rot90(img), np.rot90(room), np.rot90(icon)
            if len(pts):
                pts[:, 0], pts[:, 1] = pts[:, 1].copy(), (S - 1) - pts[:, 0]
        if random.random() < 0.5:
            img, room, icon = img[:, ::-1], room[:, ::-1], icon[:, ::-1]
            if len(pts):
                pts[:, 0] = (S - 1) - pts[:, 0]
        img = np.ascontiguousarray(img).astype(np.float32)
        # photometric
        if random.random() < 0.3:
            g = img.mean(axis=2, keepdims=True)
            img = np.repeat(g, 3, axis=2)
        if random.random() < 0.8:
            img = (img - 128.0) * random.uniform(0.75, 1.25) + 128.0 + random.uniform(-25, 25)
        if random.random() < 0.2:
            img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.3, 1.0))
        if random.random() < 0.25:
            img = img + np.random.normal(0, random.uniform(2, 8), img.shape).astype(np.float32)
        img = np.clip(img, 0, 255).astype(np.uint8)
        return img, np.ascontiguousarray(room), np.ascontiguousarray(icon), pts


# ----------------------------------------------------------------------------- model
def build_model(encoder="tu-efficientnet_b3", weights="imagenet"):
    import segmentation_models_pytorch as smp
    model = smp.Unet(encoder_name=encoder, encoder_weights=weights, classes=N_OUT,
                     decoder_attention_type="scse")
    # focal-loss prior for heatmap logits (CenterNet init)
    head = [m for m in model.segmentation_head.modules() if isinstance(m, nn.Conv2d)][-1]
    with torch.no_grad():
        head.bias[N_ROOM + N_ICON:].fill_(-2.19)
    # Some smp decoder blocks own parameters that never get gradients (e.g. the attention on
    # the missing skip of the last block). DDP aborts on those, so detect + freeze them.
    saved = {k: v.clone() for k, v in model.state_dict().items()}  # keep BN stats untouched
    model.train()
    model(torch.randn(2, 3, 64, 64)).float().sum().backward()
    for p in model.parameters():
        if p.grad is None:
            p.requires_grad_(False)
        p.grad = None
    model.load_state_dict(saved)
    return model


def split_logits(out):
    return out[:, :N_ROOM], out[:, N_ROOM:N_ROOM + N_ICON], out[:, N_ROOM + N_ICON:]


# ----------------------------------------------------------------------------- losses
def dice_loss(logits, target, n_cls, ignore=IGNORE, eps=1.0):
    valid = (target != ignore)
    t = torch.where(valid, target, torch.zeros_like(target))
    prob = logits.softmax(1) * valid.unsqueeze(1)
    oh = F.one_hot(t, n_cls).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
    dims = (0, 2, 3)
    inter = (prob * oh).sum(dims)
    card = prob.sum(dims) + oh.sum(dims)
    present = oh.sum(dims) > 0  # only classes present in the batch
    dice = (2 * inter + eps) / (card + eps)
    if present.any():
        return 1 - dice[present].mean()
    return logits.sum() * 0.0


def focal_ce(logits, target, gamma=2.0, ignore=IGNORE):
    ce = F.cross_entropy(logits, target, ignore_index=ignore, reduction="none")
    valid = (target != ignore)
    if not valid.any():
        return logits.sum() * 0.0
    pt = torch.exp(-ce)
    return (((1 - pt) ** gamma) * ce)[valid].mean()


def heat_focal(logits, gt, alpha=2.0, beta=4.0):
    keep = gt[:, 0, 0, 0] >= 0                     # samples whose source labels junctions (else target is -1)
    if not keep.any():
        return logits.sum() * 0.0
    logits, gt = logits[keep], gt[keep]
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = gt.ge(0.999).float()
    neg = 1 - pos
    pos_loss = torch.log(pred) * (1 - pred) ** alpha * pos
    neg_loss = torch.log(1 - pred) * pred ** alpha * (1 - gt) ** beta * neg
    n_pos = pos.sum().clamp(min=1.0)
    return -(pos_loss.sum() + neg_loss.sum()) / n_pos


class MultiTaskLoss(nn.Module):
    def __init__(self, w_room=1.0, w_icon=1.0, w_heat=1.0):
        super().__init__()
        self.w = (w_room, w_icon, w_heat)

    def forward(self, out, room, icon, heat):
        r, i, h = [t.float() for t in split_logits(out)]
        not_wall = room == NOT_WALL
        room = torch.where(not_wall, torch.full_like(room, IGNORE), room)
        l_room = F.cross_entropy(r, room, ignore_index=IGNORE) + dice_loss(r, room, N_ROOM)
        if not_wall.any():                          # "not a wall, room type unknown": push only P(wall) down
            p_wall = torch.softmax(r, 1)[:, ROOM_CLASSES.index("Wall")]
            l_room = l_room + (-torch.log((1 - p_wall[not_wall]).clamp(min=1e-6))).mean()
        l_icon = focal_ce(i, icon) + dice_loss(i, icon, N_ICON)
        l_heat = heat_focal(h, heat)
        total = self.w[0] * l_room + self.w[1] * l_icon + self.w[2] * l_heat
        return total, {"room": l_room.detach(), "icon": l_icon.detach(), "heat": l_heat.detach()}


# ----------------------------------------------------------------------------- metrics
def update_confusion(cm, pred, target, n_cls, ignore=IGNORE):
    valid = (target != ignore) & (target != NOT_WALL)
    idx = target[valid] * n_cls + pred[valid]
    cm += torch.bincount(idx, minlength=n_cls * n_cls).reshape(n_cls, n_cls)
    return cm


def iou_from_confusion(cm):
    cm = cm.double()
    tp = cm.diag()
    denom = cm.sum(0) + cm.sum(1) - tp
    iou = torch.where(denom > 0, tp / denom.clamp(min=1), torch.full_like(tp, float("nan")))
    return iou  # nan for classes absent from both pred and gt


class EMA:
    """Exponential moving average of weights (fp32), with warmup-aware decay."""

    def __init__(self, model, decay=0.999):
        import copy
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


class SourceWeightedSampler(torch.utils.data.Sampler):
    """Each epoch draws `num_samples` items with per-source shares (e.g. CubiCasa 0.4, ResPlan 0.25, synthetic
    angled 0.2, FloorPlanCAD 0.15), so a big source does not drown the others; sharded across DDP ranks."""

    def __init__(self, ds, weights, num_samples, rank=0, world=1, seed=0):
        self.groups = {}
        for i in range(len(ds)):
            self.groups.setdefault(ds.source_of(i), []).append(i)
        w = {k: float(weights.get(k, 0.0)) for k in self.groups}
        tot = sum(w.values()) or 1.0
        self.share = {k: v / tot for k, v in w.items() if v > 0}
        self.num_samples = num_samples - num_samples % world
        self.rank, self.world, self.seed, self.epoch = rank, world, seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        g = random.Random(self.seed + self.epoch)
        picks = []
        for k, sh in self.share.items():
            n = int(round(sh * self.num_samples))
            pool = self.groups[k]
            picks += [pool[g.randrange(len(pool))] for _ in range(n)] if n > len(pool) else g.sample(pool, n)
        g.shuffle(picks)
        picks = picks[:self.num_samples]
        picks += picks[:self.num_samples - len(picks)]
        return iter(picks[self.rank::self.world])

    def __len__(self):
        return self.num_samples // self.world


In [ ]:
%%writefile /kaggle/working/furn_src/fp_extra_fpcad.py
"""FloorPlanCAD (real CAD drawings, Voxel51 FiftyOne mirror) -> floor-plan model cache entries.

FloorPlanCAD labels drawn strokes (a wall = its two parallel lines, a door = arc + leaf). CubiCasa5K - and the
model - work with areas: a wall is the filled band, a door / window is the opening rectangle in the wall. So:
  wall   strokes closed across the wall thickness            -> room head Wall
  window hull of the window strokes (they sit in the wall)    -> icon head Window
  door   the wall gap next to the door symbol                 -> icon head Door only with teach_doors=True
         (rebuilt from strokes it is ~75% right, and CubiCasa5K / ResPlan / synthetic plans give exact door
         labels, so by default the whole door area is unknown here)
  stairs, toilet, sink, bath, appliances: hull of the symbol  -> matching icon class
  other furniture / unlabelled ink / paper                    -> icon head NoIcon
The room head only learns walls here: everything else is NOT_WALL (no room types in FloorPlanCAD), a 1 px band
around each wall and the openings are unknown (IGNORE). No junction labels.
Licence: FloorPlanCAD is CC BY-NC 4.0 (non-commercial).
"""
import os

import cv2
import numpy as np

from fp_common import ICON_CLASSES, ROOM_CLASSES, IGNORE, NOT_WALL

I = {n: i for i, n in enumerate(ICON_CLASSES)}
WALL = ROOM_CLASSES.index("Wall")
RAILING = ROOM_CLASSES.index("Railing")

DOOR = {"single_door", "double_door", "sliding_door", "folding_door", "revolving_door", "rolling_door", "door"}
WINDOW = {"window", "bay_window", "blind_window", "opening_symbol"}
ICON_MAP = {"stairs": "Stairs", "toilet": "Toilet", "squat_toilet": "Toilet", "urinal": "Toilet", "sink": "Sink",
            "bath": "Bathtub", "bath_tub": "Bathtub", "refrigerator": "ElectricalAppliance",
            "washing_machine": "ElectricalAppliance", "gas_stove": "ElectricalAppliance"}
UNSURE = {"curtain_wall", "wardrobe", "high_cabinet", "elevator", "escalator"}   # neither taught nor denied


def norm(s):
    return str(s).strip().lower().replace(" ", "_").replace("-", "_")


def _full_mask(bbox, mask, W, H):
    """FiftyOne detection (normalised bbox + mask cropped to it) -> full-size boolean stroke mask."""
    bx, by, bw, bh = [float(v) for v in bbox]
    x0, y0 = int(round(bx * W)), int(round(by * H))
    w, h = max(1, int(round(bw * W))), max(1, int(round(bh * H)))
    out = np.zeros((H, W), bool)
    if mask is None:
        return out
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m[..., 0]
    m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0
    x1, y1 = min(W, x0 + w), min(H, y0 + h)
    if x1 <= max(x0, 0) or y1 <= max(y0, 0):
        return out
    out[max(y0, 0):y1, max(x0, 0):x1] = m[max(0, -y0):y1 - y0, max(0, -x0):x1 - x0]
    return out


def _hull(m):
    ys, xs = np.nonzero(m)
    out = np.zeros(m.shape, np.uint8)
    if len(xs) >= 3:
        cv2.fillPoly(out, [cv2.convexHull(np.stack([xs, ys], 1).astype(np.int32))], 1)
    return out > 0


def convert(img_rgb, dets, ppm, teach_doors=False):
    """dets: [(label, bbox_norm, mask)], ppm: drawing pixels per metre. Returns (room, icon) label maps."""
    H, W = img_rgb.shape[:2]
    strokes = {}
    for label, bbox, mask in dets:
        strokes.setdefault(norm(label), []).append(_full_mask(bbox, mask, W, H))
    return convert_strokes(strokes, H, W, ppm, teach_doors)


def convert_strokes(strokes, H, W, ppm, teach_doors=False, solid=(), glass=()):
    """strokes: {label: [boolean stroke mask of one instance (or one drawn element)]}.
    solid: labels drawn as an outline that is all wall (columns) -> filled outline = Wall.
    glass: labels for glazing drawn along a wall line (curtain walls, glass) -> closed like a wall = Window.
    Returns (room, icon) label maps."""
    k = max(3, int(round(0.35 * ppm))) | 1                     # bridges a wall's two lines (walls < 35 cm)
    wall_st = np.zeros((H, W), bool)
    for m in strokes.get("wall", []):
        wall_st |= m
    wall = cv2.morphologyEx(wall_st.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((k, k), np.uint8)) > 0
    for lab in solid:
        for m in strokes.get(lab, []):
            wall |= _hull(m)
    room = np.full((H, W), NOT_WALL, np.uint8)
    room[wall] = WALL
    for m in strokes.get("railing", []):
        room[cv2.dilate(m.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0] = RAILING
    edge = cv2.morphologyEx(wall.astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    room[edge] = IGNORE
    icon = np.full((H, W), I["NoIcon"], np.uint8)
    for lab, ms in strokes.items():
        if lab == "wall":
            continue
        for m in ms:
            hull = _hull(m) if lab not in glass else None
            if lab in glass:
                band = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((k, k), np.uint8)) > 0
                icon[band] = I["Window"]
                room[band] = IGNORE
            elif lab in solid:
                continue
            elif lab in WINDOW:
                icon[hull] = I["Window"]
                room[hull] = IGNORE
            elif lab in DOOR:
                icon[hull] = IGNORE                            # swing area: unknown
                # the opening: wall gap bridged next to the symbol (closing wide enough for the door width)
                ys, xs = np.nonzero(hull)
                if len(xs) == 0:
                    continue
                pad = int(0.6 * ppm)
                x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad + 1, W)
                y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad + 1, H)
                loc = wall[y0:y1, x0:x1].astype(np.uint8)
                kd = int(1.5 * max(xs.max() - xs.min(), ys.max() - ys.min())) + 3   # > the opening width
                near = cv2.dilate(hull[y0:y1, x0:x1].astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
                (_, _), (_, _), ang = cv2.minAreaRect(np.stack([xs, ys], 1).astype(np.float32))
                gap = np.zeros(loc.shape, bool)
                for a in (ang, ang + 90):                      # the wall runs along one side of the swing square
                    ker = np.zeros((kd, kd), np.uint8)
                    c, dx, dy = kd // 2, np.cos(np.radians(a)) * kd / 2, np.sin(np.radians(a)) * kd / 2
                    cv2.line(ker, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))), 1, 1)
                    filled = ((cv2.morphologyEx(loc, cv2.MORPH_CLOSE, ker) > 0) & (loc == 0)).astype(np.uint8)
                    n_g, cc, stt, _ = cv2.connectedComponentsWithStats(filled)
                    for j in range(1, n_g):
                        comp = cc == j
                        if not (comp & near).any():
                            continue
                        ys2, xs2 = np.nonzero(comp)
                        (_, _), (rw, rh), _ = cv2.minAreaRect(np.stack([xs2, ys2], 1).astype(np.float32))
                        if min(rw, rh) <= 1.6 * k and max(rw, rh) >= 0.4 * ppm:   # one wall thick, door wide
                            gap |= comp
                if gap.sum() >= 4:
                    icon[y0:y1, x0:x1][gap] = I["Door"] if teach_doors else IGNORE
                    room[y0:y1, x0:x1][gap] = IGNORE
            elif lab in ICON_MAP:
                icon[hull] = I[ICON_MAP[lab]]
            elif lab in UNSURE:
                icon[hull & (icon == I["NoIcon"])] = IGNORE
    return room, icon


def build_fpcad(cache_root, max_side=1536, val_fraction=0.08, seed=0, teach_doors=False, log=print):
    """Download the FiftyOne mirror (same one the furniture notebook uses) and write cache entries."""
    import random
    import subprocess
    import sys
    from fp_render import write_entry
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fiftyone"], check=True)
    import fiftyone.utils.huggingface as fouh
    ds = fouh.load_from_hub("Voxel51/FloorPlanCAD")
    schema = ds.get_field_schema()
    field = [k for k, v in schema.items() if "Detections" in str(getattr(v, "document_type", ""))][0]
    data_dir = os.path.join(cache_root, "data")
    os.makedirs(data_dir, exist_ok=True)
    rng = random.Random(seed)
    out = {"train": [], "val": []}
    for k, s in enumerate(ds.iter_samples()):
        img = cv2.imread(s.filepath, cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        if img.ndim == 3 and img.shape[2] == 4:                # transparent PNG -> on white
            a = img[:, :, 3:4].astype(np.float32) / 255.0
            img = (img[:, :, :3].astype(np.float32) * a + 255.0 * (1 - a)).astype(np.uint8)
        rgb = cv2.cvtColor(img if img.ndim == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), cv2.COLOR_BGR2RGB)
        dets = []
        for d in (s[field].detections if s[field] else []):
            m = d.mask
            if m is None and getattr(d, "mask_path", None):
                m = cv2.imread(d.mask_path, cv2.IMREAD_GRAYSCALE)
            dets.append((d.label, d.bounding_box, m))
        ppm = max(rgb.shape[:2]) / 20.0                        # FloorPlanCAD crops are 20 m x 20 m blocks
        room, icon = convert(rgb, dets, ppm, teach_doors)
        sid = "fpcad{:05d}".format(k)
        write_entry(data_dir, sid, rgb, room, icon, max_side)
        out["val" if rng.random() < val_fraction else "train"].append({"id": sid, "source": "floorplancad"})
        if (k + 1) % 500 == 0:
            log("  floorplancad converted {}".format(k + 1))
    return out


In [ ]:
%%writefile /kaggle/working/furn_src/fp_extra_archcad.py
"""ArchCAD-400K (40K public samples, real CAD drawings of mostly public / commercial buildings) -> floor-plan model
cache entries, and -> furniture / door-type detector samples.

Each sample is an SVG of a 14 m x 14 m slice of a drawing. Every drawn element carries its class:
    <path ... semantic="single_door" instance="single_door_23"/>
(the numeric id table of the dataset card is accepted too: semantic="1" / semantic-id="1"). The SVG is rendered
here (black strokes on white) so image and labels line up exactly; then the same stroke -> area rules as
FloorPlanCAD (fp_extra_fpcad.convert_strokes) give the label maps:
    wall                       -> room head Wall
    concrete / steel column    -> filled outline = Wall (the converter projects columns as wall blocks)
    glass                      -> closed along its line = Window (ArchCAD has no separate window class)
    doors (4 kinds)            -> wall gap next to the symbol = Door only with teach_doors (else unknown)
    staircase, toilets, sinks, bathtub -> matching icon class
    axis & grid lines, beams, text, furniture, everything else -> known NOT wall (this is the lesson the
                                  model is missing: drawing notations are not walls)
Slices of structural drawings (foundation, pile, rebar) are skipped. No room types (room head = NOT_WALL).
Licence: cc-by-nc-4.0, gated on Hugging Face (accept the access agreement once with the account whose token
the notebook uses). Non-commercial use only.
"""
import os
import re
import xml.etree.ElementTree as ET

import cv2
import numpy as np

from fp_common import IGNORE
from fp_extra_fpcad import convert_strokes

HF_REPO = "jackluoluo/ArchCAD"
SLICE_M = 14.0
# the dataset card's id table
ID_NAMES = {0: "axis_grid", 1: "single_door", 2: "double_door", 3: "parent_child_door", 4: "other_door",
            5: "elevator", 6: "staircase", 7: "sink", 8: "urinal", 9: "toilet", 10: "bathtub", 11: "squat_toilet",
            12: "other_fixtures", 13: "drain", 14: "table", 15: "chair", 16: "bed", 17: "sofa", 18: "hole",
            19: "glass", 20: "wall", 21: "concrete_column", 22: "steel_column", 23: "concrete_beam",
            24: "steel_beam", 25: "parking_space", 26: "foundation", 27: "pile", 28: "rebar", 29: "fire_hydrant",
            100: "others"}
# ArchCAD name -> name the FloorPlanCAD rules know
TO_FPCAD = {"staircase": "stairs", "stair": "stairs", "bathtub": "bath_tub", "parent_child_door": "double_door",
            "other_door": "door", "concrete_column": "column", "steel_column": "column"}
SOLID = ("column",)
GLASS = ("glass",)
UNSURE = {"elevator", "hole", "other_fixtures", "drain", "fire_hydrant"}
STRUCTURAL = {"foundation", "pile", "rebar"}
# detector (furniture + door types) labels from ArchCAD
DET_MAP = {"single_door": "single_door", "double_door": "double_door", "parent_child_door": "parent_child_door",
           "table": "table", "chair": "chair", "bed": "bed", "sofa": "sofa", "sink": "sink", "toilet": "toilet",
           "squat_toilet": "squat_toilet", "urinal": "urinal", "bathtub": "bath_tub"}


def norm(s):
    s = str(s).strip().lower()
    if re.fullmatch(r"\d+", s):
        return ID_NAMES.get(int(s), "others")
    s = re.sub(r"[\s\-&]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return {"axis_and_grid": "axis_grid", "axis": "axis_grid", "grid": "axis_grid",
            "parent_child": "parent_child_door", "mother_child_door": "parent_child_door"}.get(s, s)


# ----------------------------------------------------------------------------------------- SVG -> primitives
_NUM = r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?"


def _floats(s):
    return [float(v) for v in re.findall(_NUM, s or "")]


def _mat(tr):
    """SVG transform attribute -> 3x3 matrix."""
    M = np.eye(3)
    for name, args in re.findall(r"(\w+)\s*\(([^)]*)\)", tr or ""):
        a = _floats(args)
        T = np.eye(3)
        if name == "matrix" and len(a) == 6:
            T = np.array([[a[0], a[2], a[4]], [a[1], a[3], a[5]], [0, 0, 1]])
        elif name == "translate":
            T[0, 2], T[1, 2] = a[0], (a[1] if len(a) > 1 else 0.0)
        elif name == "scale":
            T[0, 0], T[1, 1] = a[0], (a[1] if len(a) > 1 else a[0])
        elif name == "rotate":
            c, s_ = np.cos(np.radians(a[0])), np.sin(np.radians(a[0]))
            R = np.array([[c, -s_, 0], [s_, c, 0], [0, 0, 1]])
            if len(a) == 3:
                T1, T2 = np.eye(3), np.eye(3)
                T1[:2, 2], T2[:2, 2] = a[1:3], [-a[1], -a[2]]
                R = T1 @ R @ T2
            T = R
        M = M @ T
    return M


def _arc(p0, rx, ry, phi, large, sweep, p1, n=16):
    """SVG elliptical arc -> points (endpoint parameterisation, SVG spec F.6.5)."""
    if rx == 0 or ry == 0:
        return [p1]
    phi = np.radians(phi)
    c, s_ = np.cos(phi), np.sin(phi)
    dx, dy = (p0[0] - p1[0]) / 2, (p0[1] - p1[1]) / 2
    x1, y1 = c * dx + s_ * dy, -s_ * dx + c * dy
    rx, ry = abs(rx), abs(ry)
    lam = x1 ** 2 / rx ** 2 + y1 ** 2 / ry ** 2
    if lam > 1:
        rx, ry = rx * np.sqrt(lam), ry * np.sqrt(lam)
    num = rx ** 2 * ry ** 2 - rx ** 2 * y1 ** 2 - ry ** 2 * x1 ** 2
    co = np.sqrt(max(num, 0) / max(rx ** 2 * y1 ** 2 + ry ** 2 * x1 ** 2, 1e-12))
    if large == sweep:
        co = -co
    cx1, cy1 = co * rx * y1 / ry, -co * ry * x1 / rx
    cx, cy = c * cx1 - s_ * cy1 + (p0[0] + p1[0]) / 2, s_ * cx1 + c * cy1 + (p0[1] + p1[1]) / 2
    ang = lambda ux, uy, vx, vy: np.arctan2(ux * vy - uy * vx, ux * vx + uy * vy)
    t1 = ang(1, 0, (x1 - cx1) / rx, (y1 - cy1) / ry)
    dt = ang((x1 - cx1) / rx, (y1 - cy1) / ry, (-x1 - cx1) / rx, (-y1 - cy1) / ry)
    if not sweep and dt > 0:
        dt -= 2 * np.pi
    elif sweep and dt < 0:
        dt += 2 * np.pi
    ts = t1 + dt * np.linspace(0, 1, n + 1)[1:]
    return [(c * rx * np.cos(t) - s_ * ry * np.sin(t) + cx, s_ * rx * np.cos(t) + c * ry * np.sin(t) + cy) for t in ts]


def _path(d):
    """SVG path data -> list of polylines (subpaths), each [(x, y), ...]. Curves are sampled."""
    toks = re.findall(r"[MmLlHhVvCcSsQqTtAaZz]|" + _NUM, d or "")
    subs, cur, pos, start, cmd, i = [], [], (0.0, 0.0), (0.0, 0.0), None, 0
    last_ctrl = None

    def take(n):
        nonlocal i
        v = [float(t) for t in toks[i:i + n]]
        i += n
        return v
    while i < len(toks):
        if re.fullmatch(r"[A-Za-z]", toks[i]):
            cmd = toks[i]
            i += 1
            if cmd in "Zz":
                if cur:
                    cur.append(start)
                    subs.append(cur)
                cur, pos = [], start
                continue
        if cmd is None:
            break
        rel = cmd.islower()
        C = cmd.upper()
        ox, oy = pos if rel else (0.0, 0.0)
        try:
            if C == "M":
                x, y = take(2)
                pos = (x + ox, y + oy)
                if cur:
                    subs.append(cur)
                cur, start = [pos], pos
                cmd = "l" if rel else "L"
            elif C == "L":
                x, y = take(2)
                pos = (x + ox, y + oy)
                cur.append(pos)
            elif C == "H":
                (x,) = take(1)
                pos = (x + ox, pos[1])
                cur.append(pos)
            elif C == "V":
                (y,) = take(1)
                pos = (pos[0], y + oy)
                cur.append(pos)
            elif C in "CS":
                if C == "C":
                    x1, y1, x2, y2, x, y = take(6)
                    c1 = (x1 + ox, y1 + oy)
                else:
                    x2, y2, x, y = take(4)
                    c1 = (2 * pos[0] - last_ctrl[0], 2 * pos[1] - last_ctrl[1]) if last_ctrl else pos
                c2, p1 = (x2 + ox, y2 + oy), (x + ox, y + oy)
                p0 = pos
                for t in np.linspace(0, 1, 9)[1:]:
                    u = 1 - t
                    cur.append((u ** 3 * p0[0] + 3 * u * u * t * c1[0] + 3 * u * t * t * c2[0] + t ** 3 * p1[0],
                                u ** 3 * p0[1] + 3 * u * u * t * c1[1] + 3 * u * t * t * c2[1] + t ** 3 * p1[1]))
                pos, last_ctrl = p1, c2
                continue
            elif C in "QT":
                if C == "Q":
                    x1, y1, x, y = take(4)
                    c1 = (x1 + ox, y1 + oy)
                else:
                    x, y = take(2)
                    c1 = (2 * pos[0] - last_ctrl[0], 2 * pos[1] - last_ctrl[1]) if last_ctrl else pos
                p0, p1 = pos, (x + ox, y + oy)
                for t in np.linspace(0, 1, 9)[1:]:
                    u = 1 - t
                    cur.append((u * u * p0[0] + 2 * u * t * c1[0] + t * t * p1[0],
                                u * u * p0[1] + 2 * u * t * c1[1] + t * t * p1[1]))
                pos, last_ctrl = p1, c1
                continue
            elif C == "A":
                rx, ry, phi, large, sweep, x, y = take(7)
                p1 = (x + ox, y + oy)
                cur += _arc(pos, rx, ry, phi, int(large), int(sweep), p1)
                pos = p1
            else:
                i += 1
        except ValueError:
            break
        last_ctrl = None
    if cur:
        subs.append(cur)
    return [s_ for s_ in subs if len(s_) >= 1]


def _shape_polys(el, tag):
    """One SVG element -> (polylines, closed?, filled?) in its own user units."""
    g = lambda k, d=0.0: float(el.get(k, d) or d)
    if tag == "path":
        return _path(el.get("d")), False
    if tag == "line":
        return [[(g("x1"), g("y1")), (g("x2"), g("y2"))]], False
    if tag in ("polyline", "polygon"):
        v = _floats(el.get("points"))
        pts = list(zip(v[0::2], v[1::2]))
        return ([pts + pts[:1]] if tag == "polygon" and pts else [pts]), tag == "polygon"
    if tag == "rect":
        x, y, w, h = g("x"), g("y"), g("width"), g("height")
        return [[(x, y), (x + w, y), (x + w, y + h), (x, y + h), (x, y)]], True
    if tag in ("circle", "ellipse"):
        cx, cy = g("cx"), g("cy")
        rx = g("r") if tag == "circle" else g("rx")
        ry = g("r") if tag == "circle" else g("ry")
        t = np.linspace(0, 2 * np.pi, 33)
        return [list(zip(cx + rx * np.cos(t), cy + ry * np.sin(t)))], True
    return [], False


def parse_svg(path_or_text):
    """-> (elements, (minx, miny, w, h)) with elements = [(label, instance, [polyline xy arrays], is_text)].
    Labels are normalised names (norm). Coordinates are in the root user space (transforms applied)."""
    txt = open(path_or_text, encoding="utf-8", errors="ignore").read() if os.path.exists(str(path_or_text)) \
        else str(path_or_text)
    root = ET.fromstring(txt)
    vb = _floats(root.get("viewBox"))
    out = []

    def attr(el, *names):
        for n in names:
            for k, v in el.attrib.items():
                if k.split("}")[-1] == n:
                    return v
        return None

    def walk(el, M, inherited):
        tag = el.tag.split("}")[-1]
        M = M @ _mat(el.get("transform"))
        sem = attr(el, "semantic", "semantic-id", "semanticId", "semantic_id") or inherited[0]
        ins = attr(el, "instance", "instance-id", "instanceId", "instance_id") or inherited[1]
        if tag in ("path", "line", "polyline", "polygon", "rect", "circle", "ellipse"):
            polys, _closed = _shape_polys(el, tag)
            arrs = []
            for p in polys:
                if not p:
                    continue
                a = np.asarray(p, float)
                a = (M @ np.c_[a, np.ones(len(a))].T).T[:, :2]
                arrs.append(a)
            if arrs:
                out.append((norm(sem) if sem is not None else "others", ins, arrs, False))
        elif tag == "text":
            x, y = (_floats(el.get("x")) or [0.0])[0], (_floats(el.get("y")) or [0.0])[0]
            fs = el.get("font-size") or (re.findall(r"font-size:\s*([^;]+)", el.get("style", "") or "") or [""])[0]
            size = (_floats(fs) or [10.0])[0]
            s = "".join(el.itertext()).strip()
            if s:
                p = (M @ np.array([x, y, 1.0]))[:2]
                h = size * np.sqrt(abs(np.linalg.det(M[:2, :2])) or 1.0)
                out.append((norm(sem) if sem is not None else "others", ins, [np.array([p, [p[0] + h * 0.6 * len(s), p[1]]])],
                            (s, h)))
        for ch in el:
            walk(ch, M, (sem, ins))

    walk(root, np.eye(3), (None, None))
    if len(vb) == 4:
        box = tuple(vb)
    else:
        allp = np.concatenate([a for e in out for a in e[2]]) if out else np.zeros((1, 2))
        mn, mx = allp.min(0), allp.max(0)
        box = (mn[0], mn[1], max(mx[0] - mn[0], 1e-6), max(mx[1] - mn[1], 1e-6))
    return out, box


def render(elements, box, size_px, line_px=1, rng=None):
    """Draw the slice: black strokes on white, size_px on the longer side. Returns
    (rgb, {label: [mask per instance]}, {label: [point array per instance]} (for detector boxes), ppm)."""
    minx, miny, w, h = box
    s = size_px / max(w, h)
    W, H = max(8, int(round(w * s))), max(8, int(round(h * s)))
    img = np.full((H, W), 255, np.uint8)
    inst_masks, inst_pts = {}, {}
    for lab, ins, arrs, text in elements:
        key = (lab, ins if ins is not None else id(arrs))
        m = inst_masks.get(key)
        if m is None:
            m = inst_masks[key] = np.zeros((H, W), np.uint8)
        pts_all = []
        for a in arrs:
            q = np.round((a - [minx, miny]) * s).astype(np.int32)
            pts_all.append(q)
            if text:
                (st, hh) = text
                hp = max(6, int(round(hh * s)))
                scale = hp / 22.0
                cv2.putText(img, st, tuple(int(v) for v in q[0]), cv2.FONT_HERSHEY_SIMPLEX, scale, 0,
                            max(1, int(round(scale * 1.5))), cv2.LINE_AA)
                cv2.putText(m, st, tuple(int(v) for v in q[0]), cv2.FONT_HERSHEY_SIMPLEX, scale, 1,
                            max(1, int(round(scale * 1.5))), cv2.LINE_8)
                continue
            if len(q) == 1:
                q = np.vstack([q, q])
            cv2.polylines(img, [q], False, 0, line_px, cv2.LINE_AA)
            cv2.polylines(m, [q], False, 1, max(1, line_px), cv2.LINE_8)
        inst_pts.setdefault(key, []).extend(pts_all)
    strokes, points = {}, {}
    for (lab, _), m in inst_masks.items():
        strokes.setdefault(lab, []).append(m > 0)
    for (lab, _), pts in inst_pts.items():
        points.setdefault(lab, []).append(np.concatenate(pts) if pts else np.zeros((0, 2)))
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    return rgb, strokes, points, size_px / SLICE_M


def convert_sample(svg, size_px=1400, teach_doors=False, line_px=1):
    """One ArchCAD SVG -> (rgb, room, icon) or None (structural slice / nothing useful in it)."""
    elements, box = parse_svg(svg)
    labs = {e[0] for e in elements}
    if labs & STRUCTURAL or not (labs - {"others", "axis_grid"}):
        return None
    rgb, strokes, _, ppm = render(elements, box, size_px, line_px)
    H, W = rgb.shape[:2]
    fp = {}
    for lab, ms in strokes.items():
        fp.setdefault(TO_FPCAD.get(lab, lab), []).extend(ms)
    room, icon = convert_strokes(fp, H, W, ppm, teach_doors, solid=SOLID, glass=GLASS)
    for lab in UNSURE:                                        # neither taught nor denied
        for m in strokes.get(lab, []):
            ys, xs = np.nonzero(m)
            if len(xs) >= 3:
                hull = np.zeros((H, W), np.uint8)
                cv2.fillPoly(hull, [cv2.convexHull(np.stack([xs, ys], 1).astype(np.int32))], 1)
                icon[(hull > 0) & (icon == 0)] = IGNORE
    return rgb, room, icon


def detector_boxes(svg, size_px):
    """One ArchCAD SVG -> (rgb, [(detector label, 4x2 corners px)]) for the furniture / door-type detector."""
    elements, box = parse_svg(svg)
    if {e[0] for e in elements} & STRUCTURAL:
        return None
    rgb, _, points, _ = render(elements, box, size_px)
    out = []
    for lab, plist in points.items():
        if lab not in DET_MAP:
            continue
        for pts in plist:
            if len(pts) < 3:
                continue
            rect = cv2.minAreaRect(pts.astype(np.float32))
            if min(rect[1]) < 2:
                continue
            out.append((DET_MAP[lab], cv2.boxPoints(rect)))
    return rgb, out


def find_svgs(root):
    out = []
    for dp, _, fs in os.walk(root):
        out += [os.path.join(dp, f) for f in fs if f.lower().endswith(".svg")]
    return sorted(out)


def download(work_dir, token, log=print):
    """Gated HF dataset -> local folder with the SVGs (archives are unpacked)."""
    import tarfile
    import zipfile
    from huggingface_hub import snapshot_download
    d = os.path.join(work_dir, "ArchCAD")
    try:
        # the repo holds data/{svg,png,json,point,caption}.zip (2.6 GB); only the SVGs (311 MB) are needed
        snapshot_download(HF_REPO, repo_type="dataset", token=token, local_dir=d,
                          allow_patterns=["data/svg.zip", "*.svg", "README.md"])
    except Exception as e:
        raise RuntimeError("ArchCAD download failed ({}: {}). It is a gated dataset: open "
                           "https://huggingface.co/datasets/{} with the account of this HF token and accept the "
                           "access agreement (approval can take up to 3 business days).".format(type(e).__name__, e, HF_REPO))
    if not find_svgs(d):
        for dp, _, fs in os.walk(d):
            for f in fs:
                p = os.path.join(dp, f)
                if f.endswith(".zip"):
                    with zipfile.ZipFile(p) as z:
                        z.extractall(dp)
                elif f.endswith((".tar", ".tar.gz", ".tgz")):
                    with tarfile.open(p) as t:
                        t.extractall(dp)
    svgs = find_svgs(d)
    if not svgs:
        listing = [os.path.relpath(os.path.join(dp, f), d) for dp, _, fs in os.walk(d) for f in fs][:40]
        raise RuntimeError("ArchCAD: no SVG files found after download. Files seen: {}".format(listing))
    log("  ArchCAD: {} SVG slices".format(len(svgs)))
    return svgs


def probe(svgs, n=5, log=print):
    """Print what the first files contain (labels found), so a format change shows up at once."""
    import collections
    c = collections.Counter()
    for p in svgs[:n]:
        els, box = parse_svg(p)
        c.update(e[0] for e in els)
    log("  ArchCAD probe ({} files): {}".format(min(n, len(svgs)), dict(c.most_common())))
    if not (set(c) & {"wall", "single_door", "staircase", "glass"}):
        raise RuntimeError("ArchCAD: no known class names in the SVGs - the label attribute format changed: "
                           "{}".format(dict(c.most_common(15))))


def build_archcad(cache_root, work_dir, token, max_side=1536, limit=None, val_fraction=0.05, seed=0,
                  teach_doors=False, log=print):
    """Download, convert and write cache entries. Returns {"train": [...], "val": [...]}."""
    import random
    from fp_render import write_entry
    svgs = download(work_dir, token, log)
    probe(svgs, log=log)
    rng = random.Random(seed)
    rng.shuffle(svgs)
    if limit:
        svgs = svgs[:limit]
    data_dir = os.path.join(cache_root, "data")
    os.makedirs(data_dir, exist_ok=True)
    out = {"train": [], "val": []}
    skipped = 0
    for k, p in enumerate(svgs):
        try:
            got = convert_sample(p, size_px=min(max_side, 1400), teach_doors=teach_doors,
                                 line_px=1 + int(rng.random() < 0.3))
        except Exception:
            got = None
        if got is None:
            skipped += 1
            continue
        rgb, room, icon = got
        sid = "archcad{:06d}".format(k)
        write_entry(data_dir, sid, rgb, room, icon, max_side)
        out["val" if rng.random() < val_fraction else "train"].append({"id": sid, "source": "archcad"})
        if (k + 1) % 1000 == 0:
            log("  archcad converted {} / {} (skipped {})".format(k + 1, len(svgs), skipped))
    log("  archcad: {} train, {} val, {} skipped (structural / empty / unreadable)".format(
        len(out["train"]), len(out["val"]), skipped))
    return out


In [ ]:
%%writefile /kaggle/working/furn_src/train_furn.py
import json, sys
from ultralytics import YOLO
cfg = json.load(open(sys.argv[1]))
if cfg.get("resume_from"):
    YOLO(cfg["resume_from"]).train(resume=True)
else:
    YOLO(cfg["model"]).train(
        data=cfg["data"], imgsz=cfg["imgsz"], epochs=cfg["epochs"], time=cfg["hours"], batch=cfg["batch"],
        device=cfg["device"], workers=cfg["workers"], project=cfg["project"], name=cfg["name"], exist_ok=True,
        patience=60, cos_lr=True, seed=42, amp=True, plots=True,
        # line drawings: no colour jitter; any rotation / flip is valid for furniture symbols
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, degrees=180, fliplr=0.5, flipud=0.5,
        mosaic=1.0, scale=0.5, translate=0.1, close_mosaic=10)


In [ ]:
# ===================== SANITY CHECKS =====================
import urllib.request
def fail(msg): raise RuntimeError("\n\n*** SETUP PROBLEM: " + msg + " ***\n")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus))
if len(gpus) < 2: fail("Need 2 GPUs: Settings -> Accelerator -> GPU T4 x2.")
try: urllib.request.urlopen("https://huggingface.co", timeout=20)
except Exception as e: fail(f"No internet ({e}). Settings -> Internet -> On.")
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Kaggle secret not readable:", e)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if not HF_TOKEN: fail("Add Kaggle Secret HF_TOKEN (Add-ons -> Secrets) and attach it to this notebook.")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()["name"]
MODEL_REPO, DATA_REPO = f"{HF_USER}/{MODEL_REPO_NAME}", f"{HF_USER}/{DATA_REPO_NAME}"
try:
    api.create_repo(MODEL_REPO, private=True, exist_ok=True)
    api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_file(path_or_fileobj=b"Furniture OBB detector for floor plans (YOLO11m-OBB, FloorPlanCAD, CC BY-NC 4.0).\n"
                    b"Files: checkpoints/best.pt, checkpoints/last.pt, results.csv, furniture_meta.json\n",
                    path_in_repo="README.md", repo_id=MODEL_REPO, commit_message="model card")
except Exception as e: fail(f"Token cannot write to Hugging Face: {e}")
free = shutil.disk_usage("/tmp").free / 1e9
print(f"HF OK -> https://huggingface.co/{MODEL_REPO} | free disk {free:.0f} GB")
if free < 15: fail("Need ~15 GB free disk.")

In [ ]:
# ===================== DATA: HF cache -> else FloorPlanCAD via FiftyOne =====================
sys.path.insert(0, SRC_DIR)
import furn_common as F, random, tarfile, collections
CACHE_NAME = f"floorplancad_obb_{DATA_VERSION}{'_archcad' + str(ARCHCAD_LIMIT) if USE_ARCHCAD else ''}.tar"
META_PATH = os.path.join(DS_DIR, "furniture_meta.json")

def restore_cache():
    try:
        if CACHE_NAME not in api.list_repo_files(DATA_REPO, repo_type="dataset"):
            return False
        print("Restoring converted dataset from HF ...")
        p = hf_hub_download(DATA_REPO, CACHE_NAME, repo_type="dataset", token=HF_TOKEN, local_dir="/tmp/hf_dl")
        os.makedirs(DS_DIR, exist_ok=True)
        with tarfile.open(p) as t: t.extractall(DS_DIR)
        os.remove(p)
        return os.path.exists(META_PATH)
    except Exception as e:
        print("cache restore failed:", e); return False

def add_archcad(ppm, rng, stats):
    """ArchCAD door / furniture symbols, rendered at the FloorPlanCAD drawing scale (so one detector scale)."""
    import cv2, numpy as np
    import fp_extra_archcad as A
    try:
        svgs = A.download("/tmp/archcad_dl", HF_TOKEN)
        A.probe(svgs)
    except Exception as e:
        print("WARNING: ArchCAD skipped:", e); return
    rng.shuffle(svgs)
    size = int(round(A.SLICE_M * (ppm or 60.0)))
    n_img = n_obj = 0
    for k, p in enumerate(svgs[:ARCHCAD_LIMIT or None]):
        try:
            got = A.detector_boxes(p, size)
        except Exception:
            got = None
        if not got or not got[1]:
            continue
        rgb, boxes = got
        H, W = rgb.shape[:2]
        split = "val" if rng.random() < VAL_FRACTION else "train"
        stem = f"ac{k:06d}"
        cv2.imwrite(os.path.join(DS_DIR, f"images/{split}/{stem}.jpg"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        lines = []
        for lab, corners in boxes:
            lines.append(F.obb_line(F.DET_CLASSES.index(lab), np.asarray(corners), W, H))
            stats.setdefault(lab, []).append(float(max(np.linalg.norm(corners[1] - corners[0]),
                                                       np.linalg.norm(corners[2] - corners[1]))))
        open(os.path.join(DS_DIR, f"labels/{split}/{stem}.txt"), "w").write("\n".join(lines) + "\n")
        n_img += 1; n_obj += len(lines)
        if (k + 1) % 2000 == 0:
            print(f"  archcad {k+1}: {n_img} images, {n_obj} objects", flush=True)
    print(f"ArchCAD added: {n_img} images, {n_obj} objects")

def build_from_fiftyone():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fiftyone"], check=True)
    import fiftyone as fo
    import fiftyone.utils.huggingface as fouh
    print("Downloading FloorPlanCAD (Voxel51 mirror) ...")
    ds = fouh.load_from_hub("Voxel51/FloorPlanCAD")
    schema = ds.get_field_schema()
    det_fields = [k for k, v in schema.items() if "Detections" in str(getattr(v, "document_type", ""))]
    if not det_fields: raise RuntimeError(f"No Detections field found. Fields: {list(schema)}")
    field = det_fields[0]
    print(f"{len(ds)} samples, using field '{field}'")
    all_labels = collections.Counter()
    for s in ds.iter_samples():
        for d in (s[field].detections if s[field] else []):
            all_labels[F.norm_label(d.label)] += 1
    print("labels in dataset:", dict(all_labels.most_common()))
    hit = [c for c in F.DET_CLASSES if all_labels.get(c)]
    if len(hit) < 3: raise RuntimeError(f"Class names do not match. Dataset labels: {list(all_labels)}")
    missing = [c for c in F.DET_CLASSES if c not in hit]
    if missing: print("note: classes with no examples:", missing)
    for sub in ("images/train", "images/val", "labels/train", "labels/val"):
        os.makedirs(os.path.join(DS_DIR, sub), exist_ok=True)
    rng = random.Random(SEED); stats = {}; n_obj = 0; t0 = time.time()
    samples = list(ds.iter_samples())
    for k, s in enumerate(samples):
        dets = []
        for d in (s[field].detections if s[field] else []):
            mask = d.mask
            if mask is None and getattr(d, "mask_path", None):
                import cv2; mask = cv2.imread(d.mask_path, cv2.IMREAD_GRAYSCALE)
            dets.append((d.label, d.bounding_box, mask))
        split = "val" if rng.random() < VAL_FRACTION else "train"
        stem = f"{k:05d}"
        n_obj += F.write_sample(s.filepath, dets, os.path.join(DS_DIR, f"images/{split}/{stem}.jpg"),
                                os.path.join(DS_DIR, f"labels/{split}/{stem}.txt"), stats)
        if (k + 1) % 500 == 0 or k + 1 == len(samples):
            el = time.time() - t0
            print(f"  converted {k+1}/{len(samples)} ({el/(k+1)*(len(samples)-k-1)/60:.1f} min left), {n_obj} objects")
    ppm = F.dataset_px_per_m(stats)
    if USE_ARCHCAD:
        add_archcad(ppm, rng, stats)
    meta = {"classes": F.DET_CLASSES, "train_px_per_m": ppm, "data_version": DATA_VERSION,
            "counts": {c: len(v) for c, v in stats.items()}, "imgsz": IMGSZ,
            "source": "Voxel51/FloorPlanCAD" + (" + ArchCAD-400K" if USE_ARCHCAD else "")}
    json.dump(meta, open(META_PATH, "w"), indent=1)
    print("objects per class:", meta["counts"], "| drawing scale px/m:", meta["train_px_per_m"])
    print("Uploading converted dataset to HF (so future runs skip this) ...")
    tar = "/tmp/" + CACHE_NAME
    with tarfile.open(tar, "w") as t:
        for item in ("images", "labels", "furniture_meta.json"):
            t.add(os.path.join(DS_DIR, item), arcname=item)
    try:
        api.upload_file(path_or_fileobj=tar, path_in_repo=CACHE_NAME, repo_id=DATA_REPO, repo_type="dataset")
    except Exception as e:
        print("WARNING: cache upload failed (training continues):", e)
    os.remove(tar)

t0 = time.time()
if not restore_cache():
    build_from_fiftyone()
open(os.path.join(DS_DIR, "data.yaml"), "w").write(
    f"path: {DS_DIR}\ntrain: images/train\nval: images/val\nnames:\n" +
    "".join(f"  {i}: {c}\n" for i, c in enumerate(json.load(open(META_PATH))["classes"])))
meta = json.load(open(META_PATH))
n_tr = len(os.listdir(os.path.join(DS_DIR, "images/train"))); n_va = len(os.listdir(os.path.join(DS_DIR, "images/val")))
print(f"data ready in {(time.time()-t0)/60:.1f} min: {n_tr} train / {n_va} val images | scale {meta['train_px_per_m']} px/m")

In [ ]:
# ===================== RESUME (pull last.pt from HF) =====================
import torch
W_DIR = os.path.join(RUNS_DIR, RUN_NAME, "weights")
os.makedirs(W_DIR, exist_ok=True)
resume_from, finished = None, False
remote = set(api.list_repo_files(MODEL_REPO))
for f in ("checkpoints/last.pt", "checkpoints/best.pt"):
    if f in remote:
        p = hf_hub_download(MODEL_REPO, f, token=HF_TOKEN, local_dir="/tmp/hf_ck")
        shutil.copyfile(p, os.path.join(W_DIR, os.path.basename(f)))
        print("restored", f)
if "checkpoints/results.csv" in remote:
    shutil.copyfile(hf_hub_download(MODEL_REPO, "checkpoints/results.csv", token=HF_TOKEN, local_dir="/tmp/hf_ck"),
                    os.path.join(RUNS_DIR, RUN_NAME, "results.csv"))
last = os.path.join(W_DIR, "last.pt")
init_from = None
try:                                     # the detector on HF was trained for other classes: start from its weights
    rmeta = json.load(open(hf_hub_download(MODEL_REPO, "furniture_meta.json", token=HF_TOKEN, local_dir="/tmp/hf_ck")))
except Exception:
    rmeta = {}
if os.path.exists(last) and rmeta.get("data_version") != DATA_VERSION:
    best_old = os.path.join(W_DIR, "best.pt")
    init_from = "/tmp/prev_best.pt"
    shutil.copyfile(best_old if os.path.exists(best_old) else last, init_from)
    for f_ in ("last.pt", "best.pt"):
        if os.path.exists(os.path.join(W_DIR, f_)): os.remove(os.path.join(W_DIR, f_))
    rc_ = os.path.join(RUNS_DIR, RUN_NAME, "results.csv")
    if os.path.exists(rc_): os.remove(rc_)
    print(f"new classes ({rmeta.get('data_version', 'v1')} -> {DATA_VERSION}): new run starting from the previous weights")
if os.path.exists(last):
    ck = torch.load(last, map_location="cpu", weights_only=False)
    if ck.get("epoch", -1) == -1:
        finished = True
        print("Training already finished on HF. Delete checkpoints/ in the repo to train again.")
    else:
        resume_from = last
        print(f"resuming from epoch {ck['epoch'] + 1}")
    del ck
else:
    print("fresh run")

In [ ]:
# ===================== BACKGROUND UPLOADER (every new epoch -> HF) =====================
def upload_now(msg):
    files = [(os.path.join(W_DIR, "last.pt"), "checkpoints/last.pt"), (os.path.join(W_DIR, "best.pt"), "checkpoints/best.pt"),
             (os.path.join(RUNS_DIR, RUN_NAME, "results.csv"), "checkpoints/results.csv"),
             (os.path.join(RUNS_DIR, RUN_NAME, "args.yaml"), "checkpoints/args.yaml"), (META_PATH, "furniture_meta.json")]
    from huggingface_hub import CommitOperationAdd
    stage = "/tmp/hf_stage"; os.makedirs(stage, exist_ok=True)
    ops = []
    for src, dst in files:
        if os.path.exists(src):
            tmp = os.path.join(stage, dst.replace("/", "_")); shutil.copyfile(src, tmp)
            ops.append(CommitOperationAdd(path_in_repo=dst, path_or_fileobj=tmp))
    for attempt in range(5):
        try:
            api.create_commit(MODEL_REPO, operations=ops, commit_message=msg); return True
        except Exception as e:
            print(f"[hf] upload retry {attempt+1}: {e}", flush=True); time.sleep(20 * (attempt + 1))
    return False

stop_watch = threading.Event()
def watcher():
    seen = None
    while not stop_watch.wait(90):
        try:
            m = os.path.getmtime(last) if os.path.exists(last) else None
            if m and m != seen:
                time.sleep(10)                      # let the trainer finish writing
                if upload_now("epoch checkpoint"):
                    seen = m
                    rows = open(os.path.join(RUNS_DIR, RUN_NAME, "results.csv")).read().strip().splitlines()
                    print(f"[hf] checkpoint uploaded (epoch {len(rows)-1})", flush=True)
        except Exception as e:
            print("[hf] watcher:", e, flush=True)
threading.Thread(target=watcher, daemon=True).start()
print("uploader running")

In [ ]:
# ===================== TRAIN on both T4s =====================
def run_training(batch):
    hours = (NOTEBOOK_START + TIME_BUDGET_HOURS * 3600 - time.time()) / 3600 - 0.35
    if hours < 0.5: print("Not enough time left in this session."); return 0, ""
    cfg = {"model": init_from or BASE_MODEL, "data": os.path.join(DS_DIR, "data.yaml"), "imgsz": IMGSZ, "epochs": MAX_EPOCHS,
           "hours": round(hours, 2), "batch": batch, "device": [0, 1], "workers": 4, "project": RUNS_DIR,
           "name": RUN_NAME, "resume_from": resume_from}
    json.dump(cfg, open(os.path.join(SRC_DIR, "run.json"), "w"))
    print(f"training: batch {batch}, time budget {hours:.2f} h, resume={bool(resume_from)}", flush=True)
    p = subprocess.Popen([sys.executable, os.path.join(SRC_DIR, "train_furn.py"), os.path.join(SRC_DIR, "run.json")],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd="/tmp")
    tail = []
    for line in p.stdout:
        line = line.replace("\r", "\n").rstrip().split("\n")[-1]
        if not line or "it/s" in line and "100%" not in line: continue     # skip progress-bar spam
        print(line, flush=True); tail = (tail + [line])[-60:]
    return p.wait(), "\n".join(tail)

if not finished:
    rc, tail = run_training(BATCH)
    if rc != 0 and "out of memory" in tail.lower() and not os.path.exists(last):
        print("\nOut of GPU memory -> retrying with batch 8", flush=True)
        rc, tail = run_training(8)
    print("trainer exit code", rc)
stop_watch.set()
ok = upload_now("final checkpoint") if os.path.exists(last) else False
print("final upload:", ok)

In [ ]:
# ===================== RESULTS =====================
import csv
rp = os.path.join(RUNS_DIR, RUN_NAME, "results.csv")
if os.path.exists(rp):
    rows = list(csv.DictReader(open(rp)))
    rows = [{k.strip(): v for k, v in r.items()} for r in rows]
    key = [k for k in rows[0] if "mAP50-95" in k][0]; k50 = [k for k in rows[0] if k.endswith("mAP50(B)")][0]
    best = max(rows, key=lambda r: float(r[key]))
    print(f"epochs: {len(rows)} | best epoch {best['epoch']}: mAP50 {float(best[k50]):.3f}  mAP50-95 {float(best[key]):.3f}")
best_pt = os.path.join(W_DIR, "best.pt")
if os.path.exists(best_pt):
    from ultralytics import YOLO
    m = YOLO(best_pt)
    r = m.val(data=os.path.join(DS_DIR, "data.yaml"), imgsz=IMGSZ, batch=8, device=0, plots=False, verbose=True)
    shutil.copyfile(best_pt, "/kaggle/working/furniture_best.pt")
    print(f"\nmodel: https://huggingface.co/{MODEL_REPO}  (checkpoints/best.pt + furniture_meta.json)")